# LOGAN: contig-end fragments

**Draft caption.** The complete-ORF model applied unchanged to contig-end fragments: Prodigal partial genes against truncated Prodigal− ORFs, open at the 5′ or 3′ end. ROC AUC (a) and average precision (b), length alone versus length plus all four entropy features; population-weighted, out-of-fold.

**Outputs:** `../images/figL04_fragments.pdf` (vector) and `../images/figL04_fragments.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL04_fragments.tsv.gz` (4 rows; sha256 `f2938973c38d…`): complete-ORF model applied to contig-end fragments. Source on Setonix (`$LOGAN_ROOT`): `qc/analysis_models_final/models_truncated.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL04_fragments.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
r = pd.read_csv(DATA / "figL04_fragments.tsv.gz", sep="\t")
r["features"] = r.features.map({"length": "length alone", "length+aa+3di+12st+mi": "length + all four"})
r["truncation"] = r.truncation.map({"open5": "open at 5′ end", "open3": "open at 3′ end"})
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, metric, lab, letter in zip(axes, ["roc_auc", "avg_precision"], ["ROC AUC", "average precision"], "ab"):
    sns.barplot(data=r, x="truncation", y=metric, hue="features", palette=[SLOTS[0], SLOTS[1]], ax=ax)
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.3f", fontsize=9, color=INK2)
    ax.set_ylim(0, 1)
    ax.set_xlabel("")
    ax.set_ylabel(lab)
    ax.legend(fontsize=9, loc="upper left")
    panel(ax, letter)
fig.tight_layout()
save(fig, "figL04_fragments")